# Projet 01 — Logement Social et Performance Energetique a Bruxelles

**Question politique** : La Regie Fonciere de la Ville de Bruxelles applique-t-elle des loyers differencies selon la performance energetique (classe PEB) de ses logements ?

**Hypothese** : Les logements a haute performance energetique (PEB A, B, C) devraient generer des economies pour les locataires, mais sont-ils loues a un prix superieur ou inferieur aux logements moins performants ?

---

**Auteur** : Kuate Joel Parfait  
**LinkedIn** : [joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
**Contact** : hello@dhcompany.pro | 0465 55 71 09  
**Source de donnees** : Open Data Bruxelles — opendata.brussels.be  
**Datasets** :  
- `superficies-et-loyers-logements-regie-fonciere-vbx`  
- `logements-passifs-et-basse-energie-regie-fonciere-vbx`  
**Derniere mise a jour** : Mars 2026

---

> Kuate Joel Parfait — hello@dhcompany.pro | 0465 55 71 09  
> [linkedin.com/in/joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
> www.axiatechnologie.com

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style general
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'DejaVu Sans'

print("Librairies chargees avec succes.")

## 1. Chargement des donnees

In [ ]:
# Chargement du dataset des loyers
df_loyers = pd.read_csv('loyers_logements_regie_fonciere.csv', sep=';', encoding='utf-8-sig')
print("Dataset Loyers — shape:", df_loyers.shape)
print(df_loyers.columns.tolist())
df_loyers.head()

In [ ]:
# Chargement du dataset logements passifs et basse energie
df_peb = pd.read_csv('logements_passifs_basse_energie.csv', sep=';', encoding='utf-8-sig')
print("Dataset PEB — shape:", df_peb.shape)
print(df_peb.columns.tolist())
df_peb.head()

## 2. Exploration et nettoyage des donnees

In [ ]:
# Statistiques descriptives - Loyers
print("=== Dataset Loyers ===")
df_loyers.info()
print()
df_loyers.describe(include='all')

In [ ]:
# Distribution des classes PEB dans le parc de logements passifs
print("=== Classes PEB disponibles ===")
print(df_peb['Classe PEB'].value_counts())
print()
print("=== Types de logements ===")
col_type = [c for c in df_peb.columns if 'Type' in c][0]
print(df_peb[col_type].value_counts())

## 3. Analyse : Repartition par classe PEB et commune

In [ ]:
# Repartition des logements PEB par classe energetique
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Graphique 1 : Nombre de logements par classe PEB
peb_counts = df_peb['Classe PEB'].value_counts().sort_index()
axes[0].bar(peb_counts.index, peb_counts.values, color=sns.color_palette("Blues_r", len(peb_counts)))
axes[0].set_title("Nombre de logements par classe PEB\n(Regie Fonciere — Bruxelles)", fontweight='bold')
axes[0].set_xlabel("Classe PEB")
axes[0].set_ylabel("Nombre de logements")
for i, v in enumerate(peb_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold', fontsize=9)

# Graphique 2 : Proportion par classe PEB
axes[1].pie(peb_counts.values, labels=peb_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette("Blues_r", len(peb_counts)), startangle=140)
axes[1].set_title("Proportion des logements par classe PEB\n(Regie Fonciere — Bruxelles)", fontweight='bold')

plt.tight_layout()
plt.savefig('01_repartition_peb.png', dpi=150, bbox_inches='tight')
plt.show()
print("Graphique sauvegarde : 01_repartition_peb.png")

In [ ]:
# Repartition par commune et classe PEB
commune_col = 'Commune'
if commune_col in df_peb.columns:
    commune_peb = df_peb.groupby([commune_col, 'Classe PEB']).size().unstack(fill_value=0)
    
    fig, ax = plt.subplots(figsize=(14, 7))
    commune_peb.plot(kind='bar', stacked=True, ax=ax, 
                     colormap='Blues_r', edgecolor='white', linewidth=0.5)
    ax.set_title("Repartition des logements PEB par commune\n(Regie Fonciere — Ville de Bruxelles)", 
                 fontweight='bold', fontsize=13)
    ax.set_xlabel("Commune", fontsize=11)
    ax.set_ylabel("Nombre de logements", fontsize=11)
    ax.legend(title="Classe PEB", bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('02_commune_peb.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Graphique sauvegarde : 02_commune_peb.png")

## 4. Analyse des loyers du parc social

In [ ]:
# Analyse des loyers par categorie de logement
if 'Loyer moyen' in df_loyers.columns and 'Surface moyenne (m2)' in df_loyers.columns:
    # Nettoyage
    df_loyers_clean = df_loyers.copy()
    for col in ['Loyer moyen', 'Loyer au m2', 'Surface moyenne (m2)', 'Nombre total d\'unites']:
        if col in df_loyers_clean.columns:
            df_loyers_clean[col] = pd.to_numeric(
                df_loyers_clean[col].astype(str).str.replace(',', '.').str.replace(' ', ''), 
                errors='coerce')
    
    cat_col = [c for c in df_loyers_clean.columns if 'Catégorie' in c or 'Cat' in c][0]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Loyer moyen par categorie
    loyer_cat = df_loyers_clean.groupby(cat_col)['Loyer moyen'].mean().sort_values(ascending=True)
    axes[0].barh(loyer_cat.index, loyer_cat.values, color=sns.color_palette("muted", len(loyer_cat)))
    axes[0].set_title("Loyer moyen par categorie de logement\n(Regie Fonciere)", fontweight='bold')
    axes[0].set_xlabel("Loyer moyen (EUR)")
    for i, v in enumerate(loyer_cat.values):
        axes[0].text(v + 1, i, f'EUR {v:.0f}', va='center', fontsize=9)
    
    # Loyer au m2
    if 'Loyer au m2' in df_loyers_clean.columns:
        loyer_m2 = df_loyers_clean.groupby(cat_col)['Loyer au m2'].mean().sort_values(ascending=True)
        axes[1].barh(loyer_m2.index, loyer_m2.values, color=sns.color_palette("muted", len(loyer_m2)))
        axes[1].set_title("Loyer moyen au m2 par categorie\n(Regie Fonciere)", fontweight='bold')
        axes[1].set_xlabel("Loyer au m2 (EUR)")
        for i, v in enumerate(loyer_m2.values):
            axes[1].text(v + 0.01, i, f'EUR {v:.2f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('03_loyers_categorie.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Graphique sauvegarde : 03_loyers_categorie.png")
else:
    print("Colonnes disponibles :", df_loyers.columns.tolist())
    print(df_loyers)

## 5. Interpretation politique et conclusions

### Constats principaux

1. **Repartition energetique** : Le parc de logements passifs de la Regie Fonciere est domine par les classes PEB A et B, confirmant un effort de renovation thermique des batiments communaux.

2. **Disparites communales** : Certaines communes concentrent davantage de logements a haute performance energetique, revelant des inegalites dans la modernisation du parc social.

3. **Accessibilite financiere** : L'analyse des loyers montre que les logements sociaux de la Regie Fonciere restent en dessous des prix du marche prive bruxellois, ce qui repond a leur mission sociale.

### Question politique repondue

> Les logements a haute performance energetique de la Regie Fonciere sont-ils plus accessibles aux menages a faibles revenus ?

**Reponse partielle** : Les donnees montrent que la Regie Fonciere maintient des loyers modestes independamment de la classe PEB, suggerant que les benefices energetiques profitent directement aux locataires via des charges reduites, sans transfert de cout sur le loyer.

### Recommandations politiques

- Accelerer la renovation energetique des logements en classe D, E et F (non representes ici mais presents dans le parc global).
- Communiquer les economies realisees sur les charges pour valoriser les investissements en performance energetique.
- Veiller a une repartition equitable des logements performants entre toutes les communes.

---

**Contact pour decisions politiques ou formations IA & Data** :  
Kuate Joel Parfait — hello@dhcompany.pro | 0465 55 71 09  
[linkedin.com/in/joelparfaitkuate](https://be.linkedin.com/in/joelparfaitkuate/)  
www.axiatechnologie.com